In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score


In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/89fee4463f428f55d31a254924e18501a3c468c3/Data/classification_sprint/cc_approvals.data',header=None)
df.head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,b,30.83,0.000,u,g,w,v,1.250,t,t,1,f,g,00202,0,+
1,a,58.67,4.460,u,g,q,h,3.040,t,t,6,f,g,00043,560,+
2,a,24.50,0.500,u,g,q,h,1.500,t,f,0,f,g,00280,824,+
3,b,27.83,1.540,u,g,w,v,3.750,t,t,5,t,g,00100,3,+
4,b,20.17,5.625,u,g,w,v,1.710,t,f,0,f,s,00120,0,+
5,b,32.08,4.000,u,g,m,v,2.500,t,f,0,t,g,00360,0,+
6,b,33.17,1.040,u,g,r,h,6.500,t,f,0,t,g,00164,31285,+
7,a,22.92,11.585,u,g,cc,v,0.040,t,f,0,f,g,00080,1349,+
8,b,54.42,0.500,y,p,k,h,3.960,t,f,0,f,g,00180,314,+
9,b,42.50,4.915,y,p,w,v,3.165,t,f,0,t,g,00052,1442,+


In [ ]:
def data_cleaning(data, column_name):
    # --- Step 1 ---
    # Replacing ? with np.nan makes it visible to all pandas/numpy operations.
    data = data.replace('?', np.nan)

    # --- Step 2 ---
    # We loop because each column.
    for col in data.columns:
        # For numeric columns, we fill missing values with the mean.
        if data[col].dtype in ['float64', 'int64']:
            data[col] = data[col].fillna(data[col].mean())
        # For categorical (text) columns, the mean makes no sense.
        # mode()[0] gives the single most common value; we use that as the fill.
        elif data[col].dtype == 'object':
            data[col] = data[col].fillna(data[col].mode()[0])

    # Return a list of how often each unique value appears in the chosen column.
    # This lets you quickly verify the cleaning worked and inspect the distribution.
    return data[column_name].value_counts().tolist()

In [5]:
data_cleaning(df, 0)

[480, 210]

In [6]:
data_cleaning(df, 9)

[395, 295]

In [7]:
df.columns.unique()

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], dtype='int64')

In [ ]:
def data_preprocess(df):
    data = df.copy()


    # --- Step 1: Encode every text column as integers ---
    # LabelEncoder maps each unique string to a unique integer (alphabetical order).
    # We must do this before any numerical operations (scaling, splitting).
    le = LabelEncoder()
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col] = le.fit_transform(data[col])

    # --- Step 2: Drop noisy/uninformative columns, then go to NumPy ---
    # Column 13 (zip code) is effectively a unique ID — not a learnable pattern.
    # Column 11 (driver's licence) adds noise without meaningful signal here.
    data = data.drop([11, 13], axis=1)
    data = data.values  # DataFrame → NumPy array; scikit-learn expects arrays

    # --- Step 3: Separate features from the label ---
    # X = everything except the last column (the inputs)
    # y = the last column — approval status encoded as 0/1
    X = data[:, :-1] # All rows, all columns except the last one
    y = data[:, -1] # All rows, only the last column

    # --- Step 4: Scale features to [0, 1] ---
    # MinMaxScaler ensures no single feature dominates due to its raw magnitude.
    # We fit ONLY on X (full, before splitting) here for simplicity;
    # in production you would fit on X_train only to avoid any leakage.
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)

    # --- Step 5: 80/20 train-test split ---
    # random_state=42 pins the shuffle so results are reproducible across runs.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    return (X_train, y_train), (X_test, y_test)

In [11]:
(X_train, y_train), (X_test, y_test) = data_preprocess(df)
print(X_train[:1])
print(y_train[:1])
print(X_test[:1])
print(y_test[:1])

[[1.         0.25787966 0.48214286 1.         1.         0.42857143
  0.33333333 0.         0.         0.         0.         0.
  0.        ]]
[1.]
[[0.5        1.         0.05357143 0.66666667 0.33333333 0.42857143
  0.33333333 0.         0.         1.         0.02985075 0.
  0.00105   ]]
[1.]


In [12]:
def train_model(X_train, y_train):
    # Instantiate the model — solver='lbfgs' is the optimisation algorithm
    # that finds the best weights. max_iter is raised to ensure convergence.
    lm = LogisticRegression(solver='lbfgs', max_iter=1000)

    # .fit() is where learning actually happens:
    # the model sees every (feature row, label) pair and adjusts its weights
    # until the predicted probabilities best match the true training labels.
    lm.fit(X_train, y_train)

    # Return the fitted model so it can be reused for predictions and scoring.
    return lm

In [13]:
lm = train_model(X_train, y_train)
print(lm.intercept_[0])
print(lm.coef_)

1.5189304277187508
[[ 0.25123837 -0.22851285 -0.0231819   1.99522614  0.24508202 -0.29298661
  -0.08928246 -0.83827587 -3.49094192 -1.07599381 -0.83859545  0.07420654
  -1.31988688]]


### Understanding AUC-ROC Before Writing the Code

#### The core question this metric answers
Accuracy tells you "what fraction of predictions were correct?" — but it hides a problem: if 90% of applicants are approved, a model that *always* predicts "approved" gets 90% accuracy without learning anything useful.

AUC-ROC asks a sharper question: **"How well can the model rank a random approved applicant above a random rejected one?"**  
An AUC of 1.0 means perfect separation. An AUC of 0.5 means the model is no better than random guessing. AUC of 0.88 (our expected result) is quite strong.

---

#### The ROC curve — what it is

The ROC (Receiver Operating Characteristic) curve is built by sliding a decision threshold from 0 to 1:

- At each threshold, every applicant whose predicted probability ≥ threshold is classified as "approved".
- We record the **True Positive Rate** (TPR = of all actually approved, how many did we catch?) and **False Positive Rate** (FPR = of all actually rejected, how many did we wrongly approve?).
- Plotting TPR vs FPR at every threshold traces the curve.

The **AUC** (Area Under the Curve) summarises the entire curve as one number.

---

#### Why we use `predict_proba` instead of `predict`

`lm.predict(X_test)` gives hard labels — 0 or 1 based on a fixed 0.5 threshold. That throws away the model's confidence.

`lm.predict_proba(X_test)` returns a probability for **each class**:
```
[[0.73, 0.27],   # 73% chance rejected, 27% chance approved
 [0.12, 0.88],   # 12% chance rejected, 88% chance approved
 ...]
```
Column `[:, 0]` = probability of the negative class (rejected).  
Column `[:, 1]` = probability of the **positive class (approved)** ← this is what we pass to `roc_auc_score`.

Using the raw probability instead of the hard prediction lets the AUC evaluate the model's ability to *rank* applicants, not just its binary decision at one threshold — which is a much richer measure of quality.
R


### Understanding AUC-ROC Before Writing the Code

#### The problem with accuracy alone

Accuracy asks: *"what fraction of predictions were correct?"* — but it hides a trap. If 90% of applicants in our dataset were approved, a model that **always** predicts "approved" scores 90% accuracy without learning anything. We need a metric that sees through that.

AUC-ROC asks a sharper question: **"How well can the model rank a genuinely approved applicant above a genuinely rejected one?"**

---

#### What the ROC curve actually plots

The model doesn't just output "approved/rejected" — it outputs a **probability** (e.g., 0.83 = 83% chance of approval). We choose a threshold (e.g., 0.5) and classify everyone above it as approved.

The ROC curve slides that threshold from 0 → 1 and at each step records:
- **True Positive Rate (TPR / Recall):** of all truly approved applicants, what fraction did we correctly catch?
- **False Positive Rate (FPR):** of all truly rejected applicants, what fraction did we wrongly approve?

Plotting TPR vs FPR traces a curve. A perfect model hugs the top-left corner (high TPR, zero FPR). A random model traces the diagonal.

#### AUC — the single number summary

**AUC** (Area Under the Curve) collapses the whole curve into one number:
- **AUC = 1.0** → perfect — model always ranks approved above rejected
- **AUC = 0.5** → random — model has no discriminating power
- **AUC = 0.88** (our expected result) → strong — model correctly ranks an approved applicant above a rejected one 88% of the time

---

#### Why `predict_proba` and not `predict`?

`lm.predict(X_test)` gives hard labels — 0 or 1 at a fixed 0.5 threshold. That throws away the model's **confidence** and forces AUC to be computed at just one threshold.

`lm.predict_proba(X_test)` returns a full probability for each class:

```
         class 0 (rejected)   class 1 (approved)
row 0:      0.73                  0.27
row 1:      0.12                  0.88
```

- `[:, 0]` = probability of the **negative** class (rejected)
- `[:, 1]` = probability of the **positive** class (approved) ← this is the score we pass to `roc_auc_score`

Using the raw probability lets AUC evaluate across **all possible thresholds** simultaneously — a much richer picture of model quality than a single accuracy number.
E

In [14]:
### START FUNCTION
def roc_score(lm, X_test, y_test):
    # predict_proba returns shape (n_samples, 2):
    #   column 0 = P(rejected),  column 1 = P(approved)
    # We take column 1 — the positive class probability — as the ranking score.
    y_prob = lm.predict_proba(X_test)[:, 1]

    # roc_auc_score computes the AUC by comparing the true binary labels
    # against the model's confidence scores across all possible thresholds.
    return roc_auc_score(y_test, y_prob)

### END FUNCTION

In [16]:
print(roc_score(lm,X_test,y_test))

0.886344537815126


### The Four Metrics — and Why You Need All of Them

Every metric in this function comes from the same source: the **confusion matrix**. Before writing code, understand what the model can get right or wrong on a binary problem:

```
                    Predicted: Approved    Predicted: Rejected
Actual: Approved         TP (True Positive)      FN (False Negative)
Actual: Rejected         FP (False Positive)     TN (True Negative)
```

- **TP** — correctly approved a good applicant ✓
- **TN** — correctly rejected a risky applicant ✓
- **FP** — approved someone who should have been rejected (costly for the bank)
- **FN** — rejected someone who would have been fine (lost business)

Each metric below is a different way of reading that table — each one optimised for a different concern.

---

#### Accuracy — `accuracy_score(y_test, y_pred)`
```
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```
The most intuitive metric: overall fraction of correct predictions. Our expected value is **0.833** — the model gets 83 out of every 100 cases right.

**When it misleads you:** on an imbalanced dataset (e.g., 90% approved), always predicting "approved" gives 90% accuracy but is useless. That's why we need the metrics below.

---

#### Precision — `precision_score(y_test, y_pred)`
```
Precision = TP / (TP + FP)
```
Of everyone the model said "approve", what fraction truly deserved approval?

High precision = few false alarms. Expected: **0.846** — when the model says approve, it is right 84.6% of the time.

**When it matters most:** when false positives are costly — e.g., approving a card for someone who will default is an expensive mistake for the bank.

---

#### Recall — `recall_score(y_test, y_pred)`
```
Recall = TP / (TP + FN)
```
Of all applicants who truly deserved approval, what fraction did the model actually approve?

High recall = few missed approvals. Expected: **0.809** — the model catches 80.9% of all genuinely creditworthy applicants.

**When it matters most:** when false negatives are costly — e.g., in cancer screening, missing a sick patient is far worse than a false alarm. Here it represents lost business.

---

#### F1-Score — `f1_score(y_test, y_pred)`
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```
The **harmonic mean** of Precision and Recall. It punishes extreme imbalance between the two — a model with perfect precision but zero recall (or vice versa) gets an F1 of 0.

Expected: **0.827** — a balanced summary when both false approvals and missed approvals matter.

**Use F1 when** you cannot afford to sacrifice either precision or recall.

---

#### Why `predict` here, not `predict_proba`?

These four metrics all work on **hard labels** (0 or 1), not probabilities. `lm.predict(X_test)` applies the default 0.5 threshold and returns the final binary decision — exactly what accuracy, precision, recall, and F1 need to compare against `y_test`.


In [ ]:
### START FUNCTION
def scores(lm, X_test, y_test):
    # Hard predictions (0 or 1) — these metrics all compare labels, not probabilities.
    y_pred = lm.predict(X_test)

    accuracy  = accuracy_score(y_test, y_pred)   # (TP + TN) / all
    precision = precision_score(y_test, y_pred)  # TP / (TP + FP)  — quality of positives
    recall    = recall_score(y_test, y_pred)     # TP / (TP + FN)  — coverage of positives
    f1        = 2 * precision * recall / (precision + recall)         # harmonic mean of precision & recall

    return (accuracy, precision, recall, f1)

### END FUNCTION

In [17]:
(accuracy, precision, recall, f1) = scores(lm, X_test, y_test)    

print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

Accuracy: 0.833333
Precision: 0.846154
Recall: 0.808824
F1 score: 0.827068
